# 02 — Data Cleaning

Drop the patient identifier, flag clinically impossible values, and persist the cleaned (but not imputed) frame for downstream notebooks.

**Inputs**: `data/raw/heart_failure_readmission_dataset.csv` (auto-pulled from Kaggle by `load_raw()`)
**Outputs**: `data/processed/cleaned.csv`


In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

In [2]:
from src.data.load import load_raw
from src.data.clean import (
    drop_identifier,
    flag_impossible_values,
    missingness_report,
    PLAUSIBLE_RANGES,
)
from src.config import PROCESSED_DIR

In [3]:
# load the raw data so we can clean it
df = load_raw()
df.shape

(3000, 16)

In [4]:
# patient_id is just an identifier, so drop it before modeling
df = drop_identifier(df)
df.shape

(3000, 15)

In [5]:
# ranges we consider medically possible - anything outside is treated as a data-entry error
PLAUSIBLE_RANGES

{'age': (18, 110),
 'bmi': (10, 70),
 'bnp': (0, 5000),
 'sodium': (110, 160),
 'creatinine': (0, 15),
 'heart_rate': (30, 200),
 'systolic_bp': (60, 250)}

In [6]:
# turn impossible values (e.g. negative creatinine) into NaN, then re-check missingness
df = flag_impossible_values(df)
missingness_report(df)

,missing,pct
creatinine,108,3.60
bmi,91,3.03
sodium,90,3.00
heart_rate,2,0.07
age,0,0.00
gender,0,0.00
bnp,0,0.00
systolic_bp,0,0.00
ace_inhibitor,0,0.00
beta_blocker,0,0.00


In [7]:
# save the cleaned data so the next notebooks can pick it up
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
out_path = PROCESSED_DIR / 'cleaned.csv'
df.to_csv(out_path, index=False)
out_path

PosixPath('/Users/guna/projects/research/fork/AAI-500-Final-Project/data/processed/cleaned.csv')

## Notes

Imputation is deferred to `04_modeling.ipynb` so it is fit on the training split only (avoids leakage). If you need an imputed frame for EDA-style analyses, do it in a separate notebook and do **not** use it for model evaluation.
